# DEA - Nordrhein-Westfalen - Wasserverbände

For NRW, the water associations (Wasserverbände) provided additional discharge and water level data.  
We got data together with the allowance for sharing from the following "Verbände":
- Bergischer Rheinischer Wasserverband (BRW)
- Niersverband
- Aggerverband
- Ruhrverband

(Erftverband - license not yet clarified)

In [2]:
import os
import zipfile
import patoolib
from tqdm import tqdm
import pandas as pd
import numpy as np
import pyproj

from camels_de1h import get_input_path, get_nuts_id_from_provider_id, Station1h, get_metadata1h

In [5]:
metadata1h = get_metadata1h()

## Extract data

In [6]:
input_path = get_input_path("DEA") / "Verbände_NRW"

# extract zip
if not (input_path / "Q_W").exists():
    os.makedirs(input_path / "Q_W")
    
    # extract the zip file
    with zipfile.ZipFile(input_path / "Wasserverbände_data_to_Alex.zip", "r") as zip:
        zip.extractall(input_path / "Q_W")

    print("Data extracted.")
else:
    print("Data already extracted.")

Data already extracted.


## Bergischer Rheinischer Wasserverband (BRW)

Start with BRW (Bergischer Rheinischer Wasserverband).  

**Data is in MEZ / UTC+1.**

In [5]:
# make directories for BRW
input_path_brw = input_path / "Q_W" / "BRW"
os.makedirs(input_path_brw, exist_ok=True)

# extract zip
if not (input_path_brw / "W und Q").exists():
    with zipfile.ZipFile(input_path / "Q_W" / "To_Alex" / "Abfluss_Wasserstandsdaten_BRW.zip", "r") as zip:
        zip.extractall(input_path_brw)

    print("BRW data extracted.")
else:
    print("BRW data already extracted.")

BRW data extracted.


### Parse data

In [6]:
meta_brw = pd.read_csv(input_path_brw / "Stammdatentabelle.csv", sep=";", decimal=",")
meta_brw["Stationsnummer"] = meta_brw["Stationsnummer"].astype(str)

ids_brw = meta_brw["Stationsnummer"].tolist()

meta_brw

,Stationsname,Stationsnummer,X-Koordinate Messstelle [ETRS89/UTM32U],Y-Koordinate Messstelle [ETRS89/UTM32U],Höhe [DHHN92],Einzugsgebietsgröße [km2],Gewässername,Errichtungsdatum,Kommune,Kreis
0,Viehbach/Richrath Zollhaus,P002,355780.68,5667310.96,46.361,10.49,Viehbach,01.02.1967,Langenfeld,Kreis Mettmann
1,HRB Viehbach/Solingen UP,P003,359780.93,5668823.75,83.009,5.83,Viehbach,01.11.1975,Solingen,Solingen
2,Galkhausener Bach/Voßhäuschen,P005,354108.41,5666719.20,40.382,28.70,Galkhausener Bach,01.07.1972,Düsseldorf,Düsseldorf
3,Burbach/Richrath,P006,357302.81,5666583.80,50.741,3.26,Burbach,31.10.1989,Langenfeld,Kreis Mettmann
4,HRB Itter/Kuckesberg UP,P007,359354.51,5671092.25,75.148,16.82,Itter,01.07.1967,Haan,Kreis Mettmann
5,Itter/Gräfrath KW,P008,363506.27,5673464.10,146.869,6.26,Itter,01.08.1981,Solingen,Solingen
6,HRB Haaner Bach/Haan UP,P009,361613.19,5672944.52,125.149,1.61,Haaner Bach,28.01.1997,Haan,Kreis Mettmann
7,HRB Lochbach/Kasparstraße UP,P010,360368.24,5670170.62,92.358,7.23,Lochbach,14.06.1965,Solingen,Solingen
8,Lochbach/Mündung,P012,359358.22,5670945.31,76.268,8.67,Lochbach,15.07.1997,Solingen,Solingen
9,HRB Demmeltrather Bach/Solingen UP,P014,364095.60,5671761.10,171.769,1.56,Demmeltrather Bach,21.07.1997,Solingen,Solingen


In [7]:
print(f"{len(ids_brw)} stations in metadata.")

files_q = list((input_path_brw / "W und Q").glob("*Abfluss.csv"))
files_w = list((input_path_brw / "W und Q").glob("*Wasserstand.csv"))

print(f"{len(files_q)} discharge files.")
print(f"{len(files_w)} water level files.")

42 stations in metadata.
42 discharge files.
42 water level files.


In [8]:
def parse_station_data_brw(id: str, metadata: pd.DataFrame, aggregate_hourly: bool = True) -> tuple[pd.DataFrame, dict]:
    """
    Parse discharge and water level data for a given BRW station ID.  
    Returns a DataFrame containing discharge and water level data and a metadata dictionary.

    """
    # define file paths
    file_q = input_path_brw / "W und Q" / f"{id}.Abfluss.csv"
    file_w = input_path_brw / "W und Q" / f"{id}.Wasserstand.csv"

    # read data
    data_q = pd.read_csv(file_q, sep=";", skiprows=2, header=None, names=["date", "discharge_vol_obs"], parse_dates=["date"], date_format="%d.%m.%Y %H:%M")
    data_w = pd.read_csv(file_w, sep=";", skiprows=2, header=None, names=["date", "water_level_obs"], parse_dates=["date"], date_format="%d.%m.%Y %H:%M")

    # merge q and w
    data = pd.merge(data_q, data_w, on="date", how="outer").sort_values("date").reset_index(drop=True)

    # transform date time to UTC+0 (data is in UTC+1)
    data["date"] = data["date"] - pd.Timedelta(hours=1)
    data["date"] = data["date"].dt.tz_localize("UTC")

    if aggregate_hourly:
        data = data.set_index("date").resample("h", label="right", closed="right").mean().reset_index()

    # read metadata for the station
    meta = metadata[metadata["Stationsnummer"] == id].to_dict(orient="records")[0]

    return data, meta

parse_station_data_brw(ids_brw[0], meta_brw, aggregate_hourly=True)

(                            date  discharge_vol_obs  water_level_obs
 0      1967-02-01 00:00:00+00:00            0.13400          11.0000
 1      1967-02-01 01:00:00+00:00            0.13400          11.0000
 2      1967-02-01 02:00:00+00:00            0.12550          10.5000
 3      1967-02-01 03:00:00+00:00            0.11700          10.0000
 4      1967-02-01 04:00:00+00:00            0.11700          10.0000
 ...                          ...                ...              ...
 508378 2025-01-29 10:00:00+00:00            0.21125          22.9175
 508379 2025-01-29 11:00:00+00:00            0.20950          22.8325
 508380 2025-01-29 12:00:00+00:00            0.19200          22.0000
 508381 2025-01-29 13:00:00+00:00            0.19200          22.0000
 508382 2025-01-29 14:00:00+00:00            0.19200          22.0000
 
 [508383 rows x 3 columns],
 {'Stationsname': 'Viehbach/Richrath Zollhaus',
  'Stationsnummer': 'P002',
  'X-Koordinate Messstelle [ETRS89/UTM32U]': 355780.68

Run for all stations and parse metadata.

In [9]:
for id in tqdm(ids_brw):
    # get the gauge_id from the nuts_mapping, add it if it does not exist (when first running, none will exist, as we had no Verbaende data in CAMELS-DE)
    gauge_id = get_nuts_id_from_provider_id(id, "DEA", add_missing=True)

    # parse data for the station
    data, meta = parse_station_data_brw(id, meta_brw, aggregate_hourly=True)

    # parse the location
    x, y = meta["X-Koordinate Messstelle [ETRS89/UTM32U]"], meta["Y-Koordinate Messstelle [ETRS89/UTM32U]"]

    # from EPSG:4647 to EPSG:3035
    transformer = pyproj.Transformer.from_crs("epsg:25832", "epsg:3035", always_xy=True)
    easting, northing = transformer.transform(x, y)

    # from EPSG:4647 to EPSG:4326
    transformer = pyproj.Transformer.from_crs("EPSG:25832", "EPSG:4326", always_xy=True)
    lon, lat = transformer.transform(x, y)

    # if the station is not in the camelsp metadata, use the raw metadata
    station_meta = {
        "gauge_id": gauge_id,
        "provider_id": id,
        "gauge_name": meta["Stationsname"],
        "waterbody_name": meta["Gewässername"],
        "federal_state": "Nordrhein-Westfalen",
        "gauge_lon": lon,
        "gauge_lat": lat,
        "gauge_easting": easting,
        "gauge_northing": northing,
        "gauge_elev_metadata": meta["Höhe [DHHN92]"],
        "area_metadata": meta["Einzugsgebietsgröße [km2]"],
        "part_of_camelsp": False,
        }
   
    # initialize the station
    station = Station1h(gauge_id)

    # save time series data
    station.save_data(data)

    # save metadata
    station.save_metadata(**station_meta)
    
    # save raw metadata
    station.save_raw_metadata(pd.DataFrame(meta, index=[0]))
    

100%|██████████| 42/42 [04:03<00:00,  5.81s/it]


## Niersverband

**Data is in MEZ / UTC+1.**

In [10]:
# make directories for Niersverband
input_path_niers = input_path / "Q_W" / "Niersverband"
os.makedirs(input_path_niers, exist_ok=True)

# extract zip
if not (input_path_niers / "Niersverband").exists():
    with zipfile.ZipFile(input_path / "Q_W" / "To_Alex" / "Niersverband.zip", "r") as zip:
        zip.extractall(input_path_niers)

    # extract discharge
    with zipfile.ZipFile(input_path_niers / "Niersverband" / "abfluss_niersverband.zip", "r") as zip:
        zip.extractall(input_path_niers)
    # extract discharge rar
    patoolib.extract_archive(input_path_niers / "zeitreihen-abfluss.rar", outdir=input_path_niers)

    # extract water level
    with zipfile.ZipFile(input_path_niers / "Niersverband" / "wasserstand_niersverband.zip", "r") as zip:
        zip.extractall(input_path_niers)
    # extract water level rar
    patoolib.extract_archive(input_path_niers / "zeitreihen-wasserstand.rar", outdir=input_path_niers)

    print("Niersverband data extracted.")
else:
    print("Niersverband data already extracted.")

INFO patool: Extracting /home/alexd/Projekte/CAMELS_1h/Github/camels_de1h/input_data/Q_and_W/NRW_Nordrhein_Westfalen/Verbände_NRW/Q_W/Niersverband/zeitreihen-abfluss.rar ...
INFO patool: running /usr/bin/unrar x -- /home/alexd/Projekte/CAMELS_1h/Github/camels_de1h/input_data/Q_and_W/NRW_Nordrhein_Westfalen/Verbände_NRW/Q_W/Niersverband/zeitreihen-abfluss.rar
INFO patool:     with cwd='/home/alexd/Projekte/CAMELS_1h/Github/camels_de1h/input_data/Q_and_W/NRW_Nordrhein_Westfalen/Verbände_NRW/Q_W/Niersverband', input=''
INFO patool: ... /home/alexd/Projekte/CAMELS_1h/Github/camels_de1h/input_data/Q_and_W/NRW_Nordrhein_Westfalen/Verbände_NRW/Q_W/Niersverband/zeitreihen-abfluss.rar extracted to `/home/alexd/Projekte/CAMELS_1h/Github/camels_de1h/input_data/Q_and_W/NRW_Nordrhein_Westfalen/Verbände_NRW/Q_W/Niersverband'.
INFO patool: Extracting /home/alexd/Projekte/CAMELS_1h/Github/camels_de1h/input_data/Q_and_W/NRW_Nordrhein_Westfalen/Verbände_NRW/Q_W/Niersverband/zeitreihen-wasserstand.rar ..

Niersverband data extracted.


### Parse data

In [11]:
meta_niers = pd.read_csv(input_path_niers / "Niersverband" / "stammdaten-abflusspegel-nv.csv", encoding="latin1", sep=";", decimal=",")
meta_niers["ORT"] = meta_niers["ORT"].astype(str)

ids_niers = meta_niers["ORT"].tolist()

meta_niers

,NAME,ORT,BETREIBER,KOORDX,KOORDY,NULLPUNKT,GEWAESSER,STATIONIER,GEBFLAECHE,AUFGEBAUT
0,Bebericher Weg,beb,Niersverband,321784.000000,5.681277e+06,32.79,Niers,87.939,220.20,19960101
1,Bettrather Dyck,bet,Niersverband,322850.000000,5.680200e+06,33.18,Niers,89.545,199.03,19631001
2,Geldern,gel1,Niersverband,313973.000000,5.711615e+06,21.56,Niers,52.505,669.90,19830419
3,Grotendonk,gro,Niersverband,309846.629988,5.725255e+06,16.34,Kervenheimer Mühlenfleuth,3.582,63.59,19880401
4,Holtzmühle,hol,Niersverband,317737.000000,5.685045e+06,31.39,Niers,82.400,258.33,19820601
5,Kessel,kesw,Niersverband,297507.472704,5.732730e+06,9.73,Niers,13.985,1250.40,20100901
6,Klippertzmühle UW,kliuw,Niersverband,324858.000000,5.674754e+06,38.80,Niers,95.800,132.49,19681101
7,Neesendyck,kln4,Niersverband,313936.000000,5.703701e+06,25.00,Kleine Niers,5.558,607.66,19990601
8,Grafscherhof,ng1,Niersverband,305220.935084,5.724316e+06,15.11,Ottersgraben,1.041,41.04,20050825
9,Hartefelder Dyck,ng2,Niersverband,317949.000000,5.709227e+06,23.80,Sevelener Landwehr,3.765,10.76,20060410


In [12]:
print(f"{len(ids_niers)} stations in metadata.")

files_q = list((input_path_niers / "zeitreihen-abfluss").glob("*.csv"))
files_w = list((input_path_niers / "zeitreihen-wasserstand").glob("*.csv"))

print(f"{len(files_q)} discharge files.")
print(f"{len(files_w)} water level files.")

21 stations in metadata.
21 discharge files.
21 water level files.


In [13]:
def parse_station_data_niers(id: str, metadata: pd.DataFrame, aggregate_hourly: bool = True) -> tuple[pd.DataFrame, dict]:
    """
    Parse discharge and water level data for a given Niersverband station ID.  
    Returns a DataFrame containing discharge and water level data and a metadata dictionary.

    Note that some of the stations have data recorded at minutes 05, 20, 35, 50.  
    In this case, the data is interpolated to the standard 15-minute intervals (00, 15, 30, 45) 
    by linear time interpolation.

    """
    # get name from id
    gauge_name = meta_niers[meta_niers["ORT"] == id]["NAME"].values[0]
    # split and lower for file identification (save to use e.g. 'bebericher' from 'Bebericher Weg', no duplicates)
    gauge_name_split = gauge_name.split(" ")[0].lower()
    # replace umlaute
    gauge_name_split = gauge_name_split.replace("ä", "ae").replace("ö", "oe").replace("ü", "ue").replace("ß", "ss")

    # define file paths
    file_q = list((input_path_niers / "zeitreihen-abfluss").glob(f"{gauge_name_split}*.csv"))[0]
    file_w = list((input_path_niers / "zeitreihen-wasserstand").glob(f"{gauge_name_split}*.csv"))[0]

    # read data
    data_q = pd.read_csv(file_q, encoding="latin1", sep=";", decimal=",",skiprows=11, header=None, names=["date", "discharge_vol_obs"], 
                parse_dates=["date"], date_format="%d.%m.%Y %H:%M:%S", na_values=["LUECKE"])
    data_w = pd.read_csv(file_w, encoding="latin1", sep=";", decimal=",",skiprows=11, header=None, names=["date", "water_level_obs"], 
                parse_dates=["date"], date_format="%d.%m.%Y %H:%M:%S", na_values=["LUECKE"])
    
    # merge q and w
    data = pd.merge(data_q, data_w, on="date", how="outer").sort_values("date").reset_index(drop=True)

    # transform date time to UTC+0 (data is in UTC+1)
    data["date"] = data["date"] - pd.Timedelta(hours=1)
    data["date"] = data["date"].dt.tz_localize("UTC")

    # check if timestamps need interpolation (05, 20, 35, 50 pattern)
    unique_minutes = data["date"].dt.minute.unique()
    needs_interpolation = set([5, 20, 35, 50]).issubset(set(unique_minutes))
    
    if needs_interpolation:
        print(f"Station {id} ({gauge_name}): Interpolating data from minutes [05, 20, 35, 50] to [00, 15, 30, 45]")
        
        # set index for interpolation
        data = data.set_index("date")
        
        # create proper 15-minute index aligned to 00, 15, 30, 45
        start = data.index.min().floor('15min')
        end = data.index.max().ceil('15min')
        proper_index = pd.date_range(start=start, end=end, freq='15min', tz='UTC')
        
        # reindex to include both old and new timestamps, then interpolate
        data = data.reindex(data.index.union(proper_index)).interpolate(method='time').loc[proper_index]
    else:
        data = data.set_index("date")

    if aggregate_hourly:
        data = data.resample("h", label="right", closed="right").mean().reset_index()
        data.rename(columns={"index": "date"}, inplace=True)
    else:
        data = data.reset_index()
        data.rename(columns={"index": "date"}, inplace=True)

    # drop first row if discharge and water level are both NaN (happens after interpolation)
    if pd.isna(data.loc[0, "discharge_vol_obs"]) and pd.isna(data.loc[0, "water_level_obs"]):
        data = data.drop(index=0).reset_index(drop=True)

    # read metadata for the station
    meta = metadata[metadata["ORT"] == id].to_dict(orient="records")[0]

    return data, meta

parse_station_data_niers(ids_niers[0], meta_niers, aggregate_hourly=True)

Station beb (Bebericher Weg): Interpolating data from minutes [05, 20, 35, 50] to [00, 15, 30, 45]


(                            date  discharge_vol_obs  water_level_obs
 0      1995-12-13 17:00:00+00:00                NaN          29.0000
 1      1995-12-13 18:00:00+00:00                NaN          29.0000
 2      1995-12-13 19:00:00+00:00                NaN          29.0000
 3      1995-12-13 20:00:00+00:00                NaN          28.0875
 4      1995-12-13 21:00:00+00:00                NaN          28.0000
 ...                          ...                ...              ...
 253706 2024-11-21 19:00:00+00:00               3.02          87.5875
 253707 2024-11-21 20:00:00+00:00               3.02          86.3375
 253708 2024-11-21 21:00:00+00:00               3.02          85.3375
 253709 2024-11-21 22:00:00+00:00               3.02          84.0875
 253710 2024-11-21 23:00:00+00:00               3.02          83.0875
 
 [253711 rows x 3 columns],
 {'NAME': 'Bebericher Weg',
  'ORT': 'beb',
  'BETREIBER': 'Niersverband',
  'KOORDX': 321784.0,
  'KOORDY': 5681277.0,
  'NULLPUN

Run for all stations and parse metadata.

In [14]:
for id in tqdm(ids_niers):
    # get the gauge_id from the nuts_mapping, add it if it does not exist (when first running, none will exist, as we had no Verbaende data in CAMELS-DE)
    gauge_id = get_nuts_id_from_provider_id(id, "DEA", add_missing=True)

    # parse data for the station
    data, meta = parse_station_data_niers(id, meta_niers, aggregate_hourly=True)

    # parse the location
    x, y = meta["KOORDX"], meta["KOORDY"]

    # from EPSG:4647 to EPSG:3035
    transformer = pyproj.Transformer.from_crs("epsg:25832", "epsg:3035", always_xy=True)
    easting, northing = transformer.transform(x, y)

    # from EPSG:4647 to EPSG:4326
    transformer = pyproj.Transformer.from_crs("EPSG:25832", "EPSG:4326", always_xy=True)
    lon, lat = transformer.transform(x, y)

    # if the station is not in the camelsp metadata, use the raw metadata
    station_meta = {
        "gauge_id": gauge_id,
        "provider_id": id,
        "gauge_name": meta["NAME"],
        "waterbody_name": meta["GEWAESSER"],
        "federal_state": "Nordrhein-Westfalen",
        "gauge_lon": lon,
        "gauge_lat": lat,
        "gauge_easting": easting,
        "gauge_northing": northing,
        "gauge_elev_metadata": meta["NULLPUNKT"],
        "area_metadata": meta["GEBFLAECHE"],
        "part_of_camelsp": False,
        }
   
    # initialize the station
    station = Station1h(gauge_id)

    # save time series data
    station.save_data(data)

    # save metadata
    station.save_metadata(**station_meta)
    
    # save raw metadata
    station.save_raw_metadata(pd.DataFrame(meta, index=[0]))
    

  0%|          | 0/21 [00:00<?, ?it/s]

Station beb (Bebericher Weg): Interpolating data from minutes [05, 20, 35, 50] to [00, 15, 30, 45]


 14%|█▍        | 3/21 [00:15<01:35,  5.28s/it]

Station gro (Grotendonk): Interpolating data from minutes [05, 20, 35, 50] to [00, 15, 30, 45]


 29%|██▊       | 6/21 [00:32<01:18,  5.24s/it]

...

 57%|█████▋    | 12/21 [00:50<00:25,  2.80s/it]

Station ng69 (Gelinter): Interpolating data from minutes [05, 20, 35, 50] to [00, 15, 30, 45]


 76%|███████▌  | 16/21 [00:58<00:11,  2.39s/it]

Station nkw (Neukircher Weg): Interpolating data from minutes [05, 20, 35, 50] to [00, 15, 30, 45]


100%|██████████| 21/21 [01:23<00:00,  3.98s/it]


## Aggerverband

**Data is in MEZ / UTC+1.**

In [ ]:
# make directories for Aggerverband
input_path_agger = input_path / "Q_W" / "Aggerverband"
os.makedirs(input_path_agger, exist_ok=True)

# extract zip
if not (input_path_agger / "Aggerverband").exists():
    with zipfile.ZipFile(input_path / "Q_W" / "To_Alex" / "Aggerverband.zip", "r") as zip:
        zip.extractall(input_path_agger)

    print("Aggerverband data extracted.")
else:
    print("Aggerverband data already extracted.")

Aggerverband data extracted.


### Parse data

Metadata in an Excel file. The format is a little bit messed up, as two tables are in one sheet. The first rows contain information about Pegel "Brakel" (also different coordinate system), the other rows are about the other stations. We already got data for "Brakel" from LANUK NRW (and actually did not get data from Aggerverband for this station), so we can ignore this row here. The row is also red in the Excel file.

In [16]:
# read the metadata header and body
header = pd.read_excel(input_path_agger / "KI_HopeE_DE" / "AV_Stammdatentabelle.xlsx", nrows=0)
body = pd.read_excel(input_path_agger / "KI_HopeE_DE" / "AV_Stammdatentabelle.xlsx", skiprows=3, header=None, decimal=",")

# combine header and body
meta_agger = pd.DataFrame(body.values, columns=header.columns)

# some columns have to be renamed
meta_agger.rename(columns={
    "X-Koordinate Messstelle [WGS84 UTM32U]": "X-Value ETRS89/UTM Zone 32N (N-ZE)",
    "Y-Koordinate Messstelle [WGS84 UTM32U]": "Y-Value ETRS89/UTM Zone 32N (N-ZE)",
    "Pegelnullpunkt [DHHN2016/DHHN92]": "Pegelnullpunkt [DHHN20112/DHHN12/100]"
    }, inplace=True)

meta_agger["Stationsnummer"] = meta_agger["Stationsnummer"].astype(str)

ids_agger = meta_agger["Stationsnummer"].tolist()

meta_agger

,Stationsname,Stationsnummer,X-Value ETRS89/UTM Zone 32N (N-ZE),Y-Value ETRS89/UTM Zone 32N (N-ZE),Pegelnullpunkt [DHHN20112/DHHN12/100],Einzugsgebietsgröße [km2],Gewässername,Errichtungsdatum,Kommune,Kreis
0,Overath,1320017,32380005.4,5643856.0,88.5,454.0,Agger,19831101,Overath,Rhein_Berg
1,Ründeroth,1320038,32392464.0,5650772.5,135.43,323.94,Agger,20061109,Engelskichen,Oberbergischer Kreis
2,Weiershagen,1320090,32393975.9,5648248.1,147.84,137.78,Wiehl,20050106,Wiehl,Oberbergischer Kreis
3,Perke,1320015,32401665.7,5643775.1,203.58,74.2,Wiehl,19751101,Wiehl,Oberbergischer Kreis
4,Rebbelroth,1320010,32401858.6,5651235.0,189.59,110.4,Agger,19800701,Gummersbach,Oberbergischer Kreis
5,Nespen,1320012,32410251.4,5644472.0,293.24,29.4,Wiehl,19750101,Reichshof,Oberbergischer Kreis
6,Listringhausen,1320001,32403621.5,5660103.7,327.55,5.4,Genkel,19710101,Meinerzhagen,Märkischer Kreis
7,Koverstein,1320006,32405646.3,5657477.1,286.9,8.3,Agger,19711101,Gummersbach,Oberbergischer Kreis
8,Dhünnüberleitung,1320028,32381205.3,5658282.4,183.0,28.9,Kürtener Sülz,19920401,Kürten,Rhein_Berg
9,Häcksbilstein,1320029,32381214.9,5658272.0,183.14,28.9,Kürtener Sülz,19911101,Kürten,Rhein_Berg


In [17]:
print(f"{len(ids_agger)} stations in metadata.")

files_q = list((input_path_agger / "KI_HopeE_DE").glob("*Q*15min.zrx"))
files_w = list((input_path_agger / "KI_HopeE_DE").glob("*W*15min.zrx"))

print(f"{len(files_q)} discharge files.")
print(f"{len(files_w)} water level files.")

11 stations in metadata.
11 discharge files.
11 water level files.


In [18]:
def parse_station_data_agger(id: str, metadata: pd.DataFrame, aggregate_hourly: bool = True) -> tuple[pd.DataFrame, dict]:
    """
    Parse discharge and water level data for a given Aggerverband station ID.  
    Returns a DataFrame containing discharge and water level data and a metadata dictionary.

    """
    # get name from id
    gauge_name = meta_agger[meta_agger["Stationsnummer"] == id]["Stationsname"].values[0]
    gauge_name = gauge_name.replace("ä", "ae").replace("ö", "oe").replace("ü", "ue").replace("ß", "ss")

    # define file paths
    file_q = list((input_path_agger / "KI_HopeE_DE").glob(f"AV_{gauge_name[:5]}*_Q*.zrx"))[0]
    file_w = list((input_path_agger / "KI_HopeE_DE").glob(f"AV_{gauge_name[:5]}*_W*.zrx"))[0]

    # read data
    data_q = pd.read_csv(file_q, encoding="latin1", sep="\s+", skiprows=17, header=None, usecols=[0, 1], names=["date", "discharge_vol_obs"],
                        parse_dates=["date"], date_format="%Y%m%d%H%M%S")
    data_w = pd.read_csv(file_w, encoding="latin1", sep="\s+", skiprows=17, header=None, usecols=[0, 1], names=["date", "water_level_obs"], 
                        parse_dates=["date"], date_format="%Y%m%d%H%M%S")

    # merge q and w
    data = pd.merge(data_q, data_w, on="date", how="outer").sort_values("date").reset_index(drop=True)

    # transform date time to UTC+0 (data is in UTC+1)
    data["date"] = data["date"] - pd.Timedelta(hours=1)
    data["date"] = data["date"].dt.tz_localize("UTC")

    if aggregate_hourly:
        data = data.set_index("date").resample("h", label="right", closed="right").mean().reset_index()

    # read metadata for the station
    meta = meta_agger[meta_agger["Stationsnummer"] == id].to_dict(orient="records")[0]

    return data, meta

parse_station_data_agger(ids_agger[0], meta_agger, aggregate_hourly=True)

(                            date  discharge_vol_obs  water_level_obs
 0      1983-11-01 02:00:00+00:00                NaN            34.00
 1      1983-11-01 03:00:00+00:00                NaN            34.00
 2      1983-11-01 04:00:00+00:00                NaN            33.75
 3      1983-11-01 05:00:00+00:00                NaN            33.00
 4      1983-11-01 06:00:00+00:00                NaN            33.00
 ...                          ...                ...              ...
 360881 2024-12-31 19:00:00+00:00               7.67            82.00
 360882 2024-12-31 20:00:00+00:00               7.67            82.00
 360883 2024-12-31 21:00:00+00:00               7.67            82.00
 360884 2024-12-31 22:00:00+00:00               7.67            82.00
 360885 2024-12-31 23:00:00+00:00               7.67            82.00
 
 [360886 rows x 3 columns],
 {'Stationsname': 'Overath',
  'Stationsnummer': '1320017',
  'X-Value ETRS89/UTM Zone 32N (N-ZE)': 32380005.4,
  'Y-Value ETRS89/

In [22]:
for id in tqdm(ids_agger):
    # get the gauge_id from the nuts_mapping, add it if it does not exist (when first running, none will exist, as we had no Verbaende data in CAMELS-DE)
    gauge_id = get_nuts_id_from_provider_id(id, "DEA", add_missing=True)

    # parse data for the station
    data, meta = parse_station_data_agger(id, meta_agger, aggregate_hourly=True)

    # parse the location
    x, y = meta["X-Value ETRS89/UTM Zone 32N (N-ZE)"], meta["Y-Value ETRS89/UTM Zone 32N (N-ZE)"]

    # from EPSG:4647 to EPSG:3035
    transformer = pyproj.Transformer.from_crs("epsg:4647", "epsg:3035", always_xy=True)
    easting, northing = transformer.transform(x, y)

    # from EPSG:4647 to EPSG:4326
    transformer = pyproj.Transformer.from_crs("EPSG:4647", "EPSG:4326", always_xy=True)
    lon, lat = transformer.transform(x, y)

    # if the station is not in the camelsp metadata, use the raw metadata
    station_meta = {
        "gauge_id": gauge_id,
        "provider_id": id,
        "gauge_name": meta["Stationsname"],
        "waterbody_name": meta["Gewässername"],
        "federal_state": "Nordrhein-Westfalen",
        "gauge_lon": lon,
        "gauge_lat": lat,
        "gauge_easting": easting,
        "gauge_northing": northing,
        "gauge_elev_metadata": meta["Pegelnullpunkt [DHHN20112/DHHN12/100]"],
        "area_metadata": meta["Einzugsgebietsgröße [km2]"],
        "part_of_camelsp": False,
        }
   
    # initialize the station
    station = Station1h(gauge_id)

    # save time series data
    station.save_data(data)

    # save metadata
    station.save_metadata(**station_meta)
    
    # save raw metadata
    station.save_raw_metadata(pd.DataFrame(meta, index=[0]))
    

100%|██████████| 11/11 [00:57<00:00,  5.20s/it]


## Ruhrverband

**Data is in MEZ / UTC+1.**

In [26]:
# make directories for Ruhrverband
input_path_ruhr = input_path / "Q_W" / "Ruhrverband"
os.makedirs(input_path_ruhr, exist_ok=True)

# extract zip
if not (input_path_ruhr / "Ruhrverband").exists():
    with zipfile.ZipFile(input_path / "Q_W" / "To_Alex" / "Datenlieferung_KI-HopE-De_Ruhrverband.zip", "r") as zip:
        zip.extractall(input_path_ruhr)

    # extract discharge zip
    with zipfile.ZipFile(input_path_ruhr / "q_data_ruhrverband.zip", "r") as zip:
        zip.extractall(input_path_ruhr)
    # extract water level zip
    with zipfile.ZipFile(input_path_ruhr / "w_data_ruhrverband.zip", "r") as zip:
        zip.extractall(input_path_ruhr)

    print("Ruhrverband data extracted.")
else:
    print("Ruhrverband data already extracted.")

Ruhrverband data extracted.


### Parse data

In [45]:
meta_ruhr_q = pd.read_csv(input_path_ruhr / "Stammdatentabelle_Ruhrverband_Q.csv", sep=";", decimal=",").sort_values("Stationsname").reset_index(drop=True)
meta_ruhr_w = pd.read_csv(input_path_ruhr / "Stammdatentabelle_Ruhrverband_W.csv", sep=";", decimal=",").sort_values("Stationsname").reset_index(drop=True)

# check if they are equal
if meta_ruhr_q.equals(meta_ruhr_w):
    meta_ruhr = meta_ruhr_q
else:
    raise ValueError("Ruhrverband Q and W metadata do not match!")

meta_ruhr["Stationsnummer"] = meta_ruhr["Stationsnummer"].astype(str)

ids_ruhr = meta_ruhr["Stationsnummer"].tolist()

meta_ruhr

,Stationsname,Stationsnummer,X.Koordinate.Messstelle..WGS84.,Y.Koordinate.Messstelle..WGS84.,Pegelnullpunkt..DHHN92.,Einzugsgebietsgröße..km2.,Gewässername,Errichtungsdatum,Kommune,Kreis
0,Ahausen,2766495000100,7.955005,51.140256,234.763,359.50,Bigge,NaN,Finnentrop,NaN
1,Amecke,2761885000100,7.948589,51.298159,283.758,28.71,Sorpe,NaN,Sundern,NaN
2,Attendorn,2766491000100,7.900813,51.116509,251.924,332.23,Bigge,NaN,Attendorn,NaN
3,Bamenohl,2766390000100,7.982855,51.162522,233.999,453.09,Lenne,NaN,Finnentrop,NaN
4,Börlinghausen,2766465000100,7.779974,51.077716,327.034,47.98,Lister,NaN,Meinerzhagen,NaN
5,Endorf1,2761831000100,8.042010,51.301043,293.260,26.07,Röhr,NaN,Sundern,NaN
6,Essen-Werden,2769730000200,6.999371,51.383857,42.500,4336.55,Ruhr,NaN,Essen,NaN
7,Fröndenberg,2765190000100,7.691043,51.472239,113.002,1914.47,Ruhr,NaN,Iserlohn,NaN
8,Fürwigge,2766811000100,7.688532,51.150280,412.256,4.61,Verse,NaN,Meinerzhagen,NaN
9,Günne,2762715000100,8.047048,51.492891,175.087,440.14,Möhne,NaN,Möhnesee,NaN


In [47]:
print(f"{len(ids_ruhr)} stations in metadata.")

files_q = list((input_path_ruhr / "Q_aufbereitet").glob("*.csv"))
files_w = list((input_path_ruhr / "W_aufbereitet").glob("*.csv"))

print(f"{len(files_q)} discharge files.")
print(f"{len(files_w)} water level files.")

41 stations in metadata.
41 discharge files.
41 water level files.


In [89]:
def parse_station_data_ruhr(id: str, metadata: pd.DataFrame, aggregate_hourly: bool = True) -> tuple[pd.DataFrame, dict]:
    """
    Parse discharge and water level data for a given Ruhrverband station ID.  
    Returns a DataFrame containing discharge and water level data and a metadata dictionary.

    """
    # define file paths
    file_q = list((input_path_ruhr / "Q_aufbereitet").glob(f"*{id}*.csv"))[0]
    file_w = list((input_path_ruhr / "W_aufbereitet").glob(f"*{id}*.csv"))[0]

    data_q = pd.read_csv(file_q, skiprows=2, sep=";", decimal=",", na_values=["---"], header=None, names=["date", "discharge_vol_obs"], 
                        parse_dates=["date"], date_format="%d.%m.%Y %H:%M:%S")
    data_w = pd.read_csv(file_w, skiprows=2, sep=";", decimal=",", na_values=["---"], header=None, names=["date", "water_level_obs"], 
                        parse_dates=["date"], date_format="%d.%m.%Y %H:%M:%S")

    # merge q and w
    data = pd.merge(data_q, data_w, on="date", how="outer").sort_values("date").reset_index(drop=True)

    # transform date time to UTC+0 (data is in UTC+1)
    data["date"] = data["date"] - pd.Timedelta(hours=1)
    data["date"] = data["date"].dt.tz_localize("UTC")

    if aggregate_hourly:
        data = data.set_index("date").resample("h", label="right", closed="right").mean().reset_index()

    # read metadata for the station
    meta = metadata[metadata["Stationsnummer"] == id].to_dict(orient="records")[0]

    return data, meta

parse_station_data_ruhr(ids_ruhr[0], meta_ruhr, aggregate_hourly=True)

(                            date  discharge_vol_obs  water_level_obs
 0      1990-11-01 00:00:00+00:00               4.03             90.0
 1      1990-11-01 01:00:00+00:00               4.03             90.0
 2      1990-11-01 02:00:00+00:00               4.03             90.0
 3      1990-11-01 03:00:00+00:00               4.03             90.0
 4      1990-11-01 04:00:00+00:00               4.03             90.0
 ...                          ...                ...              ...
 299515 2024-12-31 19:00:00+00:00               8.30            136.0
 299516 2024-12-31 20:00:00+00:00               8.30            136.0
 299517 2024-12-31 21:00:00+00:00               8.30            136.0
 299518 2024-12-31 22:00:00+00:00               8.30            136.0
 299519 2024-12-31 23:00:00+00:00               8.30            136.0
 
 [299520 rows x 3 columns],
 {'Stationsname': 'Ahausen',
  'Stationsnummer': '2766495000100',
  'X.Koordinate.Messstelle..WGS84.': 7.95500548258283,
  'Y.Koor

In [90]:
for id in tqdm(ids_ruhr):
    # get the gauge_id from the nuts_mapping, add it if it does not exist (when first running, none will exist, as we had no Verbaende data in CAMELS-DE)
    gauge_id = get_nuts_id_from_provider_id(id, "DEA", add_missing=True)

    # parse data for the station
    data, meta = parse_station_data_ruhr(id, meta_ruhr, aggregate_hourly=True)

    # parse the location
    lon, lat = meta["X.Koordinate.Messstelle..WGS84."], meta["Y.Koordinate.Messstelle..WGS84."]

    # from EPSG:4326 to EPSG:3035
    transformer = pyproj.Transformer.from_crs("epsg:4326", "epsg:3035", always_xy=True)
    easting, northing = transformer.transform(lon, lat)

    # if the station is not in the camelsp metadata, use the raw metadata
    station_meta = {
        "gauge_id": gauge_id,
        "provider_id": id,
        "gauge_name": meta["Stationsname"],
        "waterbody_name": meta["Gewässername"],
        "federal_state": "Nordrhein-Westfalen",
        "gauge_lon": lon,
        "gauge_lat": lat,
        "gauge_easting": easting,
        "gauge_northing": northing,
        "gauge_elev_metadata": meta["Pegelnullpunkt..DHHN92."],
        "area_metadata": meta["Einzugsgebietsgröße..km2."],
        "part_of_camelsp": False,
        }
   
    # initialize the station
    station = Station1h(gauge_id)

    # save time series data
    station.save_data(data)

    # save metadata
    station.save_metadata(**station_meta)
    
    # save raw metadata
    station.save_raw_metadata(pd.DataFrame(meta, index=[0]))
    

100%|██████████| 41/41 [03:35<00:00,  5.26s/it]


## Erftverband

**Data is in MEZ / UTC+1.**

Erftverband does not have Pegel IDs!

In [7]:
# make directories for Erftverband
input_path_erft = input_path / "Q_W" / "Erftverband"
os.makedirs(input_path_erft, exist_ok=True)

# extract zip
if not (input_path_erft / "Erftverband").exists():
    with zipfile.ZipFile(input_path / "Q_W" / "To_Alex" / "Daten_Erftverband.zip", "r") as zip:
        zip.extractall(input_path_erft)

    print("Erftverband data extracted.")
else:
    print("Erftverband data already extracted.")

Erftverband data extracted.


### Parse data

In [8]:
meta_erft = pd.read_excel(input_path_erft / "HvZ" / "stammdaten_ev_pegel_20250204.xlsx", decimal=",")

# Erftverband does not use station IDs, so we use the gauge names instead
meta_erft["ORT"] = meta_erft["ORT"].astype(str).str.split("_").str[1]
names_erft = meta_erft["ORT"].tolist()

meta_erft

,ORT,KOORDX,KOORDY,KOORDX_ALT,KOORDY_ALT,GEWAESSER,KOMMUNE,KREIS,BETREIBER,LEITNIVPEG,NULLPUNKT,EZG [km²],Kommentar
0,Anstel,2550340,5659246,340076,5659655,Gillbach,Rommerskirchen,Neuss,Erftverband,2009,58.103,52.8,NaN
1,Bedburg,2540734,5649923,330100,5650731,Erft,Bedburg,Rhein-Erft-Kreis,Erftverband,2013,56.449,1429.5,"seit 2015, direkte Abflussmessung (RQ30), wind..."
2,Eicherscheid,2554920,5599823,342241,5600102,Erft,Bad Muenstereifel,Euskirchen,Erftverband,2007,312.206,42.7,NaN
3,Essig,2563034,5613927,350917,5613864,"Steinbach, Jungbach",Swisttal,Rhein-Sieg-Kreis,Erftverband,2009,157.886,41.1,Pegel fällt im Sommer trocken
4,Fuessenich,2543517,5618079,331587,5618803,Neffelbach,Zuelpich,Euskirchen,Erftverband,2009,153.268,16.2,NaN
5,Gill,2548290,5654764,337846,5655260,Gillbach,Rommerskirchen,Neuss,Erftverband,2009,67.598,24.5,NaN
6,Gymnich,2553697,5634281,342415,5634577,Erft,Erftstadt,Rhein-Erft-Kreis,Erftverband,2009,84.944,894.2,"seit 2010, direkte Abflussmessung (SLD, Werte ..."
7,Hausweiler,2557479,5620580,345637,5620736,Erft,Weilerswist,Euskirchen,Erftverband,2013,125.774,259.6,NaN
8,Horchheim,2558083,5621879,346293,5622009,Erft,Weilerswist,Euskirchen,Erftverband,2013,119.464,282.0,NaN
9,Horrem,2549927,5640705,338909,5641148,Kleine Erft,Kerpen,Rhein-Erft-Kreis,Erftverband,2009,71.950,NaN,ab 2009


In [9]:
print(f"{len(names_erft)} stations in metadata.")

files_q = list((input_path_erft / "HvZ").glob("*q.csv"))
files_w = list((input_path_erft / "HvZ").glob("*w.csv"))

print(f"{len(files_q)} discharge files.")
print(f"{len(files_w)} water level files.")

14 stations in metadata.
14 discharge files.
14 water level files.


In [10]:
def parse_station_data_erft(gauge_name: str, metadata: pd.DataFrame, aggregate_hourly: bool = True) -> tuple[pd.DataFrame, dict]:
    """
    Parse discharge and water level data for a given Erftverband station ID.  
    Returns a DataFrame containing discharge and water level data and a metadata dictionary.

    """
    # replace umlaute
    gauge_name = gauge_name.replace("ä", "ae").replace("ö", "oe").replace("ü", "ue").replace("ß", "ss")

    # define file paths
    file_q = list((input_path_erft / "HvZ").glob(f"Pegel_{gauge_name}_q.csv"))[0]
    file_w = list((input_path_erft / "HvZ").glob(f"Pegel_{gauge_name}_w.csv"))[0]

    data_q = pd.read_csv(file_q, sep=";", decimal=",", header=None, usecols=[2,3,4], names=["date", "time", "discharge_vol_obs"])
    data_q["date"] = pd.to_datetime(data_q["date"] + " " + data_q["time"], format="%d.%m.%Y %H:%M:%S")
    data_q = data_q.drop(columns=["time"])

    data_w = pd.read_csv(file_w, sep=";", decimal=",", header=None, usecols=[2,3,4], names=["date", "time", "water_level_obs"])
    data_w["date"] = pd.to_datetime(data_w["date"] + " " + data_w["time"], format="%d.%m.%Y %H:%M:%S")
    data_w = data_w.drop(columns=["time"])

    # merge q and w
    data = pd.merge(data_q, data_w, on="date", how="outer").sort_values("date").reset_index(drop=True)

    # transform date time to UTC+0 (data is in UTC+1)
    data["date"] = data["date"] - pd.Timedelta(hours=1)
    data["date"] = data["date"].dt.tz_localize("UTC")

    if aggregate_hourly:
        data = data.set_index("date").resample("h", label="right", closed="right").mean().reset_index()

    # read metadata for the station
    meta = metadata[metadata["ORT"] == gauge_name].to_dict(orient="records")[0]

    return data, meta

parse_station_data_erft(names_erft[0], meta_erft, aggregate_hourly=True)

(                            date  discharge_vol_obs  water_level_obs
 0      1999-10-31 23:00:00+00:00           0.141082        53.804100
 1      1999-11-01 00:00:00+00:00           0.142691        53.884530
 2      1999-11-01 01:00:00+00:00           0.142156        53.857788
 3      1999-11-01 02:00:00+00:00           0.146204        54.060212
 4      1999-11-01 03:00:00+00:00           0.151106        54.304305
 ...                          ...                ...              ...
 219164 2024-10-31 19:00:00+00:00           0.414488        72.949060
 219165 2024-10-31 20:00:00+00:00           0.410930        72.824650
 219166 2024-10-31 21:00:00+00:00           0.401812        72.505830
 219167 2024-10-31 22:00:00+00:00           0.377787        71.665802
 219168 2024-10-31 23:00:00+00:00           0.340565        70.364323
 
 [219169 rows x 3 columns],
 {'ORT': 'Anstel',
  'KOORDX': 2550340,
  'KOORDY': 5659246,
  'KOORDX_ALT': 340076,
  'KOORDY_ALT': 5659655,
  'GEWAESSER': 'Gill

In [11]:
for name in tqdm(names_erft):
    # get the gauge_id from the nuts_mapping, add it if it does not exist (when first running, none will exist, as we had no Verbaende data in CAMELS-DE)
    gauge_id = get_nuts_id_from_provider_id(name.lower(), "DEA", add_missing=True)

    # parse data for the station
    data, meta = parse_station_data_erft(name, meta_erft, aggregate_hourly=True)

    # parse the location
    x, y = meta["KOORDX_ALT"], meta["KOORDY_ALT"]

    # from EPSG:4647 to EPSG:3035
    transformer = pyproj.Transformer.from_crs("epsg:25832", "epsg:3035", always_xy=True)
    easting, northing = transformer.transform(x, y)

    # from EPSG:4647 to EPSG:4326
    transformer = pyproj.Transformer.from_crs("EPSG:25832", "EPSG:4326", always_xy=True)
    lon, lat = transformer.transform(x, y)

    # if the station is not in the camelsp metadata, use the raw metadata
    station_meta = {
        "gauge_id": gauge_id,
        "provider_id": name.lower(), # use lowercase name as provider_id
        "gauge_name": meta["ORT"],
        "waterbody_name": meta["GEWAESSER"],
        "federal_state": "Nordrhein-Westfalen",
        "gauge_lon": lon,
        "gauge_lat": lat,
        "gauge_easting": easting,
        "gauge_northing": northing,
        "gauge_elev_metadata": meta["NULLPUNKT"],
        "area_metadata": meta["EZG [km²]"],
        "part_of_camelsp": False,
        }
   
    # initialize the station
    station = Station1h(gauge_id)

    # save time series data
    station.save_data(data)

    # save metadata
    station.save_metadata(**station_meta)
    
    # save raw metadata
    station.save_raw_metadata(pd.DataFrame(meta, index=[0]))
    

100%|██████████| 14/14 [01:30<00:00,  6.49s/it]
